# 04 The baseline

Before I trust the random forest I need something to compare it with. The baseline is **logistic regression** on the same 14
numbers. Next to it two guesses that never look at a wafer: **guess by share** (what luck scores) and **always yes**
(why I compare with F1 and not recall alone).

Saves `output/baseline.joblib`.

In [1]:
import numpy as np
import joblib # for saving the model to a file
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score # sklearn does the scoring so i dont have to

import wafer

f = np.load("output/features.npz")
# X = the 14 numbers per wafer, y = the 8 answers (0 or 1)
X_train, y_train = f["X_train"], f["y_train"]
X_val, y_val = f["X_val"], f["y_val"]

## 1. How I score

- **all 8 right**: the share of wafers where all 8 answers are right. `accuracy_score` counts exactly that.
- **recall**: of the wafers that really have the defect, how many it finds.
- **F1 per defect**: high only when it finds the defect (recall) and is right when it says yes (precision).

The scores go into `scores`, so I can put them side by side in step 5.

In [2]:
scores = {}   # name -> (F1 per defect, all 8 right)


def score(name, predicted):
    all_right = accuracy_score(y_val, predicted) # with 8 answers per wafer this is the share where all 8 are right
    # average=None gives 8 scores, one per defect, instead of 1 average
    # zero_division=0 so it doesnt warn when it never says yes to a defect
    recall = recall_score(y_val, predicted, average=None, zero_division=0)
    f1 = f1_score(y_val, predicted, average=None, zero_division=0)
    scores[name] = (f1, all_right)

    print(f"{name}: all 8 right {all_right * 100:.1f}%")
    for i in range(8):
        print(f"  {wafer.DEFECTS[i]}: recall {recall[i]:.2f}, F1 {f1[i]:.2f}")

# guess by share
# per defect, say yes on a random part of the wafers as big as that defect's share in training.
# `y_train.mea(axis=0)` is the average of every column, and a column of 0s and 1s averages to the share of 1s.
shares = y_train.mean(axis=0)
rng = np.random.default_rng(42) # 42 = seed, same random numbers every run
# random number 0-1 for every answer, if its under the share its a yes. astype(int) makes True/False into 1/0
guess = (rng.random(y_val.shape) < shares).astype(int)
score("Guess by share", guess)

Guess by share: all 8 right 2.4%
  Center: recall 0.33, F1 0.32
  Donut: recall 0.33, F1 0.33
  Edge-Loc: recall 0.34, F1 0.34
  Edge-Ring: recall 0.33, F1 0.32
  Loc: recall 0.49, F1 0.50
  Near-Full: recall 0.00, F1 0.00
  Scratch: recall 0.48, F1 0.48
  Random: recall 0.05, F1 0.05


**What to look for:** recall and F1 about the same as the share of each defect. That's luck, the floor.

## 2. Always yes

In [3]:
always_yes = np.ones_like(y_val) # same shape as y_val but all 1s
score("Always yes", always_yes)

Always yes: all 8 right 0.0%
  Center: recall 1.00, F1 0.49
  Donut: recall 1.00, F1 0.49
  Edge-Loc: recall 1.00, F1 0.49
  Edge-Ring: recall 1.00, F1 0.49
  Loc: recall 1.00, F1 0.66
  Near-Full: recall 1.00, F1 0.01
  Scratch: recall 1.00, F1 0.66
  Random: recall 1.00, F1 0.04


**What to look for:** recall 1.00 on every defect, but a low F1 and 0% all 8 right. Recall alone can be bought by
saying yes to everything.

## 3. The baseline: logistic regression

Logistic regression gives every one of the 14 numbers a weight, adds them up and says yes above a cutoff. It answers
one yes/no question, so `MultiOutputClassifier` makes 8 of them, one per defect.

In [4]:
# logistic regression can only do 1 yes/no, MultiOutputClassifier makes 8 of them (one per defect)
# max_iter = how many tries it gets to find the weights, default 100 is sometimes not enough
baseline = MultiOutputClassifier(LogisticRegression(max_iter=1000))
baseline.fit(X_train, y_train) # fit = train it
score("Logistic regression", baseline.predict(X_val))

Logistic regression: all 8 right 30.5%
  Center: recall 0.81, F1 0.85
  Donut: recall 0.84, F1 0.87
  Edge-Loc: recall 0.00, F1 0.00
  Edge-Ring: recall 0.95, F1 0.92
  Loc: recall 0.79, F1 0.83
  Near-Full: recall 0.95, F1 0.93
  Scratch: recall 0.55, F1 0.60
  Random: recall 0.12, F1 0.18


**What to look for:** the defects with a low F1

## 5. Side by side (F1)

In [5]:
# :<12 and :>22 pad the text so the columns line up. end="" means stay on the same line
print(f"{'':<12}", end="")
for name in scores:
    print(f"{name:>22}", end="")
print()

for i in range(8):
    print(f"{wafer.DEFECTS[i]:<12}", end="")
    for name in scores:
        print(f"{scores[name][0][i]:>22.2f}", end="") # [0] is the F1s, [i] is this defect
    print()

print(f"{'All 8 right':<12}", end="")
for name in scores:
    print(f"{scores[name][1] * 100:>21.1f}%", end="")
print()

                    Guess by share            Always yes   Logistic regression
Center                        0.32                  0.49                  0.85
Donut                         0.33                  0.49                  0.87
Edge-Loc                      0.34                  0.49                  0.00
Edge-Ring                     0.32                  0.49                  0.92
Loc                           0.50                  0.66                  0.83
Near-Full                     0.00                  0.01                  0.93
Scratch                       0.48                  0.66                  0.60
Random                        0.05                  0.04                  0.18
All 8 right                   2.4%                  0.0%                 30.5%


## 6. Save

In [6]:
joblib.dump(baseline, "output/baseline.joblib")
print("Saved output/baseline.joblib")

Saved output/baseline.joblib
